In [27]:
!pip install sentence_transformers
!pip install torch_geometric

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\jaden\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\jaden\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [28]:
# load json
import json
from sentence_transformers import SentenceTransformer
import json
import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
from torch_geometric.data import Data, InMemoryDataset
print("import ok")

import ok


In [29]:
with open("wiki_data_yearly_triples.json", 'r') as f:
    data = json.load(f)

In [30]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6828.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
# get global node set
global_nodes = set()

for key in data.keys():
    for triple in data[key]:
        global_nodes.add(triple["head"])
        global_nodes.add(triple["tail"])
        
global_nodes = list(global_nodes)
print(len(global_nodes))

10834


In [32]:
# get global_rel set
global_rel = set()

for key in data.keys():
    for triple in data[key]:
        global_rel.add(triple["relation"])
        
global_rel = list(global_rel)
print(len(global_rel))

23


In [33]:
# index mappings for nodes and rels
id_to_node = {}
id_to_rel = {}

node_to_id = {}
rel_to_id = {}

for i, node in enumerate(global_nodes):
    id_to_node[i] = node
    node_to_id[node] = i

for i, rel in enumerate(global_rel):
    id_to_rel[i] = rel
    rel_to_id[rel] = i

In [34]:
def embed(texts, batch_size=64):
    # this will become more advanced as we encorperate more and more into the embedding
    # EX: emb = name_emb + description_emb + class_emb
    #    - or concat for more representation
    embs = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return embs.astype(np.float32)

In [41]:
import json
import os
from typing import Any, Callable, Dict, List, Optional

import numpy as np
import torch
from torch_geometric.data import Data, InMemoryDataset


class GlobalTemporalTextKGDataset(InMemoryDataset):
    def __init__(
        self,
        root: str,
        start_year: int,
        entity_to_id: Dict[str, int],
        relation_to_id: Dict[str, int],
        embed_fn: Callable[[List[str]], np.ndarray],
        json_filename: str,
        dataset_filename: str,
        transform=None,
        pre_transform=None,
        pre_filter=None,
    ):
        self.json_filename = json_filename
        self.dataset_filename = dataset_filename
        self.entity_to_id = entity_to_id
        self.relation_to_id = relation_to_id
        self.embed_fn = embed_fn
        self.start = start_year

        # Filled after loading processed file
        self.x_global: Optional[torch.Tensor] = None
        self.id_to_entity: Optional[Dict[int, str]] = None

        super().__init__(root, transform, pre_transform, pre_filter)

        obj = torch.load(self.processed_paths[0], weights_only=False)
        self.data = obj["data"]
        self.slices = obj["slices"]
        self.x_global = obj["x_global"]
        self.id_to_entity = obj["id_to_entity"]

    # get file names
    @property
    def raw_file_names(self) -> List[str]:
        return [self.json_filename]

    @property
    def processed_file_names(self) -> List[str]:
        return [self.dataset_filename]

    def download(self):
        pass

    # get node embeddings
    def _build_global_node_features(self) -> torch.Tensor:
        if len(self.entity_to_id) == 0:
            return torch.empty((0, 0), dtype=torch.float)

        # get num of nodes
        max_id = max(self.entity_to_id.values())
        num_nodes = max_id + 1

        # reverse mapping
        id_to_entity = {idx: text for text, idx in self.entity_to_id.items()}

        # make sure ids are contiguous
        missing = [i for i in range(num_nodes) if i not in id_to_entity]
        if missing:
            raise ValueError(
                f"entity_to_id must use contiguous ids from 0..{num_nodes-1}. "
                f"Missing ids: {missing[:10]}"
            )

        # order entities
        entity_texts_ordered = [id_to_entity[i] for i in range(num_nodes)]

        # get embeddings
        embeddings = self.embed_fn(entity_texts_ordered)
        embeddings = np.asarray(embeddings, dtype=np.float32)

        # check for emb errors
        if embeddings.shape[0] != num_nodes:
            raise ValueError(
                f"embed_fn returned {embeddings.shape[0]} embeddings for "
                f"{num_nodes} entities"
            )

        return torch.tensor(embeddings, dtype=torch.float)

    def process(self):
        # Load raw temporal triples
        with open(self.raw_paths[0], "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        # Build global node feature matrix once
        x_global = self._build_global_node_features()
        num_nodes = x_global.size(0)

        # reverse mapping
        id_to_entity = {idx: text for text, idx in self.entity_to_id.items()}

        data_list = []

        # iterate through years
        for year in range(self.start, 2020):
            triples = raw_data[str(year)]

            src = []
            dst = []
            rels = []
            active_nodes = set()

            for triple in triples:
                head_text = triple["head"]
                tail_text = triple["tail"]
                rel_text = triple["relation"]

                if head_text not in self.entity_to_id:
                    raise KeyError(f"Head entity not found in entity_to_id: {head_text}")

                if tail_text not in self.entity_to_id:
                    raise KeyError(f"Tail entity not found in entity_to_id: {tail_text}")

                if rel_text not in self.relation_to_id:
                    raise KeyError(f"Relation not found in relation_to_id: {rel_text}")

                h_id = self.entity_to_id[head_text]
                t_id = self.entity_to_id[tail_text]
                r_id = self.relation_to_id[rel_text]

                src.append(h_id)
                dst.append(t_id)
                rels.append(r_id)

                active_nodes.add(h_id)
                active_nodes.add(t_id)

            edge_index = (
                torch.tensor([src, dst], dtype=torch.long)
                if len(src) > 0
                else torch.empty((2, 0), dtype=torch.long)
            )

            edge_type = (
                torch.tensor(rels, dtype=torch.long)
                if len(rels) > 0
                else torch.empty((0,), dtype=torch.long)
            )

            active_nodes = (
                torch.tensor(sorted(active_nodes), dtype=torch.long)
                if len(active_nodes) > 0
                else torch.empty((0,), dtype=torch.long)
            )

            data = Data(
                edge_index=edge_index,
                edge_type=edge_type,
                year=torch.tensor([year], dtype=torch.long),
                active_nodes=active_nodes,
                num_nodes=num_nodes,
            )

            if self.pre_filter is not None and not self.pre_filter(data):
                continue

            if self.pre_transform is not None:
                data = self.pre_transform(data)

            data_list.append(data)

        data, slices = self.collate(data_list)

        torch.save(
            {
                "data": data,
                "slices": slices,
                "x_global": x_global,
                "id_to_entity": id_to_entity,
            },
            self.processed_paths[0],
        )

In [42]:
dataset = GlobalTemporalTextKGDataset(
    root="data/wikidata",
    start_year=1800,
    json_filename="wiki_data_yearly_triples.json",
    dataset_filename="simple_dataset_1800.pt",
    entity_to_id=node_to_id,
    relation_to_id=rel_to_id,
    embed_fn=embed,
)

Processing...


FileNotFoundError: [Errno 2] No such file or directory: 'data\\wikidata\\raw\\wiki_data_yearly_triples.json'

In [ ]:
# generate torch geometric dataset